# P3 AAA — RepDWNet + WDice on Kaggle

One notebook: select Early/Late/All in the first config cell. Uses the successful P2/DPP/LIP flow: clone pinned source → detect PhenoBench input → preflight/smoke → isolated benchmarks → original P2 trainer on T4×2 → one artifacts ZIP. AAA-Now uses explicit fixed Tri-3 MaxBlurPool. WSC is offline.


In [ ]:
# FIRST CONFIG CELL: choose the experiment and source here.
REPO_URL = 'https://github.com/FUAVX/RPD_Reproduce.git'
GIT_REF = 'f/down-aaa'
SOURCE_COMMIT = ''  # optional full commit SHA reachable from GIT_REF
EXPERIMENT_MODE = 'ex1'  # ex1 = Early, ex2 = Late, ex3 = All
AAA_VARIANT = 'aaa_tri3'  # fixed audited AAA-Now topology
RUN_TAG = 'run1'  # use a new tag for a separate repeat
DATASET_DIR = ''  # empty = auto-detect attached PhenoBench Kaggle input
NUM_WORKERS = 2
DEVICES = 2  # P2 runtime override; YAML default remains 1
WORKING_ROOT = '/kaggle/working'

EXPERIMENTS = {
    'ex1': ('P3_AAA_Ex1_Early', [1, 2]),
    'ex2': ('P3_AAA_Ex2_Late', [3, 4]),
    'ex3': ('P3_AAA_Ex3_All', [1, 2, 3, 4]),
}
if EXPERIMENT_MODE not in EXPERIMENTS:
    raise ValueError(f'Unknown EXPERIMENT_MODE: {EXPERIMENT_MODE}')
if AAA_VARIANT != 'aaa_tri3':
    raise NotImplementedError(f'{AAA_VARIANT} is not implemented. Implement and test the variant before selecting it.')
if DEVICES != 2:
    raise ValueError('This P2-matched protocol requires 2 devices.')
SCENARIO, EXPECTED_STAGES = EXPERIMENTS[EXPERIMENT_MODE]

RUN_BENCHMARKS = True  # disposable subprocesses, never warm-start the campaign
RUN_OFFLINE_ANALYSIS = False  # optional; may also run later using ZIP checkpoints
ANALYSIS_MAX_IMAGES = None  # None = all validation images; deterministic filename order


In [ ]:
# Kaggle runtime dependencies. Kaggle supplies torch/CUDA; do not reinstall torch.
!nvidia-smi
# Lightning 1.9.5 uses np.Inf; keep NumPy 1.x as specified in P2 requirements.
!pip install -q 'numpy<2' 'pytorch-lightning==1.9.5' 'torchmetrics==0.11.4' 'setuptools<70' tensorboard matplotlib pandas tabulate timm scikit-image tifffile albumentations pyyaml rich


In [ ]:
import json
import pathlib
import shutil
import subprocess
import yaml

WORKING_DIR = pathlib.Path(WORKING_ROOT)
REPO_DIR = WORKING_DIR / 'RPD_P2'
RUN_DIR = WORKING_DIR / f'{SCENARIO}_{AAA_VARIANT}_{RUN_TAG}'
if RUN_DIR.exists():
    raise FileExistsError(f'Output already exists: {RUN_DIR}. Choose a new RUN_TAG.')

def run(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run(['git', 'clone', '--branch', GIT_REF, '--single-branch', REPO_URL, str(REPO_DIR)])
if SOURCE_COMMIT:
    run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR)
run(['git', 'rev-parse', '--verify', 'HEAD'], cwd=REPO_DIR)

scenario_path = REPO_DIR / 'configs' / 'p3_scenarios' / f'{SCENARIO}.yaml'
required_paths = [scenario_path, REPO_DIR / 'test_wdice.py', REPO_DIR / 'test_aaa.py',
                  REPO_DIR / 'test_aaa_analysis.py', REPO_DIR / 'test_aaa_evaluation.py',
                  REPO_DIR / 'test_dpp_evaluation.py', REPO_DIR / 'scripts/aaa_evaluation.py',
                  REPO_DIR / 'scripts/dpp_evaluation.py', REPO_DIR / 'models/rpdnet/aaa.py',
                  REPO_DIR / 'callbacks/training_cost.py', REPO_DIR / 'tests/reference/aaa/blurpool.py',
                  REPO_DIR / 'models/rpdnet/downsampling.py', REPO_DIR / 'scripts/benchmark_aaa.py',
                  REPO_DIR / 'scripts/analyze_aaa.py', REPO_DIR / 'test_config_callback.py',
                  REPO_DIR / 'test_tensorboard_logging.py', REPO_DIR / 'utils/tensorboard_compat.py']
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Branch does not contain the committed P3 AAA files: ' + ', '.join(missing))

def is_phenobench_root(path):
    return (
        (path / 'train' / 'images').is_dir()
        and (path / 'train' / 'semantics').is_dir()
        and (path / 'val' / 'images').is_dir()
        and (path / 'val' / 'semantics').is_dir()
    )

if DATASET_DIR:
    dataset_dir = pathlib.Path(DATASET_DIR)
else:
    kaggle_input = pathlib.Path('/kaggle/input')
    candidates = [
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset',
        kaggle_input / 'phenobench-dataset',
    ]
    if kaggle_input.is_dir():
        attached_inputs = [p for p in kaggle_input.iterdir() if p.is_dir()]
        candidates.extend(p / 'PhenoBench' for p in attached_inputs)
        candidates.extend(attached_inputs)
    dataset_dir = next((p for p in candidates if is_phenobench_root(p)), None)
if dataset_dir is None or not is_phenobench_root(dataset_dir):
    raise FileNotFoundError('Cannot find a complete PhenoBench root in /kaggle/input. Attach the PhenoBench dataset, or set DATASET_DIR to its root containing train/images, train/semantics, val/images, and val/semantics.')

print(f'[OK] source ref: {subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()}')
print(f'[OK] dataset: {dataset_dir}')
print(f'[OK] train images: {len(list((dataset_dir / "train" / "images").glob("*")))}')


In [ ]:
# Resolve the Kaggle-specific configuration without modifying the cloned repository.
RUN_DIR.mkdir(parents=True, exist_ok=True)
with scenario_path.open(encoding='utf-8') as stream:
    cfg = yaml.safe_load(stream)

cfg['data']['path_to_dataset'] = str(dataset_dir)
cfg['data']['num_workers'] = NUM_WORKERS
active_cfg_path = RUN_DIR / f'active_config_{SCENARIO}.yaml'
with active_cfg_path.open('w', encoding='utf-8') as stream:
    yaml.safe_dump(cfg, stream, sort_keys=False)

assert cfg['train']['loss'] == 'weighted_dice'
assert cfg['loss']['dice']['lambda'] == 1.0
assert cfg['loss']['dice']['epsilon'] == 1e-6
assert cfg['train']['class_weights'] == [1.47, 5.06, 10.02]
assert cfg['train']['max_epoch'] == 100
assert cfg['val']['check_val_every_n_epoch'] == 5
assert cfg['train']['batch_size'] == 4
assert cfg['train']['early_stopping']['mode'] == 'none'
assert cfg['backbone']['downsampling']['variant'] == AAA_VARIANT
assert cfg['backbone']['downsampling']['stages'] == EXPECTED_STAGES
assert {k: cfg['backbone']['downsampling'][k] for k in ('filt_size','pad_type','stride','pad_off')} == {
    'filt_size': 3, 'pad_type': 'reflect', 'stride': 2, 'pad_off': 0}
# Match against the source P2 config to catch unintended recipe drift.
import copy
base_cfg = yaml.safe_load((REPO_DIR / 'configs/p2_scenarios/P2_Ex7_RepDWNet_WDice-Loss.yaml').read_text())
base_cfg['data']['path_to_dataset'] = str(dataset_dir)
base_cfg['data']['num_workers'] = NUM_WORKERS
inherited = copy.deepcopy(cfg)
del inherited['backbone']['downsampling']
inherited['experiment'] = base_cfg['experiment']
assert inherited == base_cfg, 'P3 differs from P2 beyond experiment metadata/downsampling.'
print(yaml.safe_dump({
    'experiment': cfg['experiment'],
    'backbone': cfg['backbone'],
    'loss': cfg['loss'],
}, sort_keys=False))

assert cfg['train']['precision'] == 32
runtime_cfg = copy.deepcopy(cfg)
runtime_cfg['train']['devices'] = DEVICES
runtime_cfg['val']['devices'] = 1  # same CLI override behavior as WDice/DPP
runtime_cfg_path = RUN_DIR / 'runtime_config.yaml'
runtime_cfg_path.write_text(yaml.safe_dump(runtime_cfg, sort_keys=False), encoding='utf-8')


In [ ]:
# The training recipe uses actual T4 x2. CPU tests do not count as GPU smoke.
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != DEVICES:
    raise RuntimeError('Select GPU T4 x2 in Kaggle settings before running.')
gpu_names = [torch.cuda.get_device_name(i) for i in range(DEVICES)]
if not all('T4' in name for name in gpu_names):
    raise RuntimeError(f'AAA campaign hardware is T4 x2; found {gpu_names}')
print('GPUs:', gpu_names)
for script in ['test_tensorboard_logging.py', 'test_wdice.py', 'test_dpp.py',
               'test_aaa.py', 'test_aaa_analysis.py', 'test_aaa_evaluation.py',
               'test_dpp_evaluation.py', 'test_config_callback.py']:
    run(['python', script], cwd=REPO_DIR)

# Actual trainer + TensorBoard + checkpoint + resume + buffer synchronization.
for mode, (scenario, stages) in EXPERIMENTS.items():
    smoke = ['python', 'test_aaa.py', '--smoke', '--config',
             str(REPO_DIR / 'configs/p3_scenarios' / f'{scenario}.yaml'),
             '--output-dir', str(RUN_DIR / 'smoke' / mode),
             '--accelerator', 'gpu', '--devices', str(DEVICES)]
    run(smoke, cwd=REPO_DIR)
    run(smoke + ['--resume'], cwd=REPO_DIR)
run(['python', 'test_aaa.py', '--native-smoke', '--accelerator', 'gpu',
     '--output-dir', str(RUN_DIR / 'smoke/native')], cwd=REPO_DIR)

# Separate processes own all benchmark models/optimizer/RNG/BN states.
if RUN_BENCHMARKS:
    run(['python', 'scripts/benchmark_aaa.py', '--mode', 'inference',
         '--output-dir', str(RUN_DIR / 'benchmark')], cwd=REPO_DIR)
    run(['python', '-m', 'torch.distributed.run', '--standalone',
         '--nproc_per_node', str(DEVICES), 'scripts/benchmark_aaa.py',
         '--mode', 'training', '--output-dir', str(RUN_DIR / 'benchmark')], cwd=REPO_DIR)

TRAIN_LOG_DIR = RUN_DIR / 'training_output'
train_command = [
    'python', 'multi_metric_train.py', '--config', str(active_cfg_path),
    '--export_dir', str(TRAIN_LOG_DIR), '--devices', str(DEVICES),
]
# Fresh process seeds the original P2 recipe; no benchmark state is loaded.
run(train_command, cwd=REPO_DIR)


In [ ]:
# Optional offline WSC on each selected checkpoint policy. Full image normalized once.
import hashlib
if RUN_OFFLINE_ANALYSIS:
    for policy in ('best_weedIoU', 'best_mIoU'):
        candidates = sorted((TRAIN_LOG_DIR / 'checkpoints').glob(f'*{policy}*.ckpt'))
        if len(candidates) != 1:
            raise RuntimeError(f'Expected one {policy} checkpoint, found {candidates}')
        command = ['python', 'scripts/analyze_aaa.py', '--config', str(runtime_cfg_path),
                   '--checkpoint', str(candidates[0]), '--dataset-dir', str(dataset_dir),
                   '--output-dir', str(RUN_DIR / 'aaa_analysis' / policy), '--device', 'cuda']
        if ANALYSIS_MAX_IMAGES is not None:
            command += ['--max-images', str(ANALYSIS_MAX_IMAGES)]
        run(command, cwd=REPO_DIR)

# Same single-ZIP output flow as WDice/DPP/LIP.
import importlib.metadata
source_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
base_yaml = REPO_DIR / 'configs/p2_scenarios/P2_Ex7_RepDWNet_WDice-Loss.yaml'
manifest = {
    'experiment': SCENARIO, 'mode': EXPERIMENT_MODE, 'variant': AAA_VARIANT, 'run_tag': RUN_TAG,
    'source_repository': REPO_URL, 'requested_ref': GIT_REF, 'requested_commit': SOURCE_COMMIT,
    'source_commit': source_commit, 'dataset_dir_at_runtime': str(dataset_dir),
    'base_yaml': str(base_yaml.relative_to(REPO_DIR)),
    'base_yaml_sha256': hashlib.sha256(base_yaml.read_bytes()).hexdigest(),
    'seed': cfg['seed'], 'resolved_config': runtime_cfg,
    'aaa_implementation': {
        'reference_repository': 'https://github.com/adobe/antialiased-cnns',
        'reference_commit': 'b27a34a26f3ab039113d44d83c54d0428598ac9c',
        'operator': 'Max2/s1/p0/ceilFalse -> ReflectionPad1 -> fixed Tri3 grouped Conv/s2',
        'settings': cfg['backbone']['downsampling'], 'filter_storage': 'persistent buffer, no Parameters',
        'backbone_initialization': 'no random draws in AAA constructor; common tensors match Control at the same seed',
        'custom_cuda': False,
    },
    'runtime_overrides': {'train.devices': DEVICES, 'val.devices': 1},
    'train_command': train_command, 'yaml_default_devices': cfg['train']['devices'],
    'batch_size_per_process': cfg['train']['batch_size'],
    'global_batch_size': cfg['train']['batch_size'] * DEVICES * cfg['train']['accumulate_grad_batches'],
    'gpu_names': gpu_names, 'cuda_version': torch.version.cuda,
    'versions': {name: importlib.metadata.version(name) for name in
                 ('torch', 'torchvision', 'pytorch-lightning', 'torchmetrics', 'numpy', 'PyYAML')},
    'benchmark_requested': RUN_BENCHMARKS,
    'benchmark_weights': 'controlled untrained disposable models; not campaign checkpoints',
    'offline_analysis_requested': RUN_OFFLINE_ANALYSIS, 'analysis_max_images': ANALYSIS_MAX_IMAGES,
    'wsc_graph': 'unfused_eval',
    'wsc_preprocessing': 'SingleImageNormalizer on native1024 once, then five1008 crops',
    'deployment_certified': False, 'deployment_note': 'strict trained-checkpoint parity remains a separate post-training audit',
    'status': 'completed',
}
(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
required_outputs = [TRAIN_LOG_DIR / 'checkpoints/last.ckpt', TRAIN_LOG_DIR / 'val_history.csv',
                    TRAIN_LOG_DIR / 'training_cost.json', runtime_cfg_path]
if RUN_BENCHMARKS:
    required_outputs += [RUN_DIR/'benchmark/benchmark_inference.json', RUN_DIR/'benchmark/benchmark_training.json']
missing = [str(path) for path in required_outputs if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing artifacts: ' + ', '.join(missing))
archive = shutil.make_archive(str(WORKING_DIR / f'{SCENARIO}_{AAA_VARIANT}_{RUN_TAG}_artifacts'),
                              'zip', root_dir=RUN_DIR)
print(f'[DONE] Download this Kaggle output: {archive}')
print(f'[DONE] Source commit: {source_commit}')
